In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# Carregando o arquivo
df = pd.read_csv('../data/spotify_user_behavior_realistic_50000_rows.csv')




ModuleNotFoundError: No module named 'seaborn'

In [19]:
#Recolhendo amostras
df.sample(20)


,user_id,country,age,signup_date,subscription_type,subscription_status,months_inactive,inactive_3_months_flag,ad_interaction,ad_conversion_to_subscription,music_suggestion_rating_1_to_5,avg_listening_hours_per_week,favorite_genre,most_liked_feature,desired_future_feature,primary_device,playlists_created,avg_skips_per_day
32657,32658,USA,38,2021-10-13,Free,Active,0,0,No,No,3,4.20,Bollywood,Lyrics,Social Listening,Mobile,5,7
16784,16785,India,39,2024-10-25,Free,Active,1,0,Yes,No,4,7.61,K-Pop,Discover Weekly,Concert Alerts,Desktop,5,6
5560,5561,Mexico,41,2020-07-18,Free,Inactive,3,1,Yes,Yes,5,12.28,Hip-Hop,Radio,Better AI Recommendations,Desktop,10,14
6253,6254,Australia,52,2019-02-23,Premium Family,Active,0,0,No,No,3,3.74,Country,Discover Weekly,Social Listening,Mobile,7,11
4721,4722,Australia,16,2022-01-29,Free,Active,2,0,Yes,No,4,8.35,Electronic,Daily Mix,Concert Alerts,Mobile,10,12
20307,20308,Australia,37,2019-01-05,Free,Inactive,3,1,No,No,1,10.46,Electronic,AI DJ,Social Listening,Tablet,7,10
34941,34942,Brazil,28,2022-06-26,Student,Active,2,0,No,No,2,12.96,Indie,Radio,Better AI Recommendations,Mobile,15,5
28788,28789,Brazil,28,2019-03-07,Free,Active,1,0,No,No,4,13.22,Jazz,Offline Mode,HiFi Audio,Mobile,6,16
7695,7696,Australia,19,2025-06-02,Premium Duo,Active,2,0,No,No,4,9.13,Pop,Podcasts,HiFi Audio,Tablet,11,6
15341,15342,Italy,21,2022-11-11,Premium Duo,Active,2,0,No,No,5,11.95,Pop,Daily Mix,Better AI Recommendations,Car System,6,16


In [17]:
#Sumário dos dados
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 18 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   user_id                         50000 non-null  int64  
 1   country                         50000 non-null  str    
 2   age                             50000 non-null  int64  
 3   signup_date                     50000 non-null  str    
 4   subscription_type               50000 non-null  str    
 5   subscription_status             50000 non-null  str    
 6   months_inactive                 50000 non-null  int64  
 7   inactive_3_months_flag          50000 non-null  int64  
 8   ad_interaction                  50000 non-null  str    
 9   ad_conversion_to_subscription   50000 non-null  str    
 10  music_suggestion_rating_1_to_5  50000 non-null  int64  
 11  avg_listening_hours_per_week    50000 non-null  float64
 12  favorite_genre                  50000 non-n

In [14]:
#Estatísticas descritivas
df.describe()

,user_id,age,months_inactive,inactive_3_months_flag,music_suggestion_rating_1_to_5,avg_listening_hours_per_week,playlists_created,avg_skips_per_day
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,25000.500000,38.010280,1.533020,0.222460,3.644100,9.988986,8.002680,10.025920
std,14433.901067,12.984989,1.952082,0.415903,1.114424,3.968927,2.831571,3.165579
min,1.000000,16.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
25%,12500.750000,27.000000,0.000000,0.000000,3.000000,7.280000,6.000000,8.000000
50%,25000.500000,38.000000,1.000000,0.000000,4.000000,9.980000,8.000000,10.000000
75%,37500.250000,49.000000,2.000000,0.000000,5.000000,12.680000,10.000000,12.000000
max,50000.000000,60.000000,18.000000,1.000000,5.000000,26.250000,23.000000,25.000000


In [29]:

#Linhas duplicadas
df.duplicated().sum()

np.int64(0)

In [37]:

colunas = [
    "avg_listening_hours_per_week",
    "avg_skips_per_day",
    "playlists_created",
    "months_inactive",
    "age"
]

resultados = []

for col in colunas:
    
    # Q1, Q3 e IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    # limites
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    # filtro de outliers
    outliers = df[(df[col] < limite_inferior) | (df[col] > limite_superior)]
    sem_outliers = df[(df[col] >= limite_inferior) & (df[col] <= limite_superior)]

    # métricas
    qtd_outliers = len(outliers)
    total = len(df)
    perc_outliers = (qtd_outliers / total) * 100

    media_com = df[col].mean()
    media_sem = sem_outliers[col].mean()

    # decisão automática
    if perc_outliers < 1:
        decisao = "Manter (dados limpos)"
    elif perc_outliers < 5:
        decisao = "OK (monitorar)"
    elif perc_outliers < 10:
        decisao = "Avaliar impacto"
    else:
        decisao = "Investigar forte presença de outliers"

    resultados.append({
        "Coluna": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Limite Inferior": limite_inferior,
        "Limite Superior": limite_superior,
        "Qtd Outliers": qtd_outliers,
        "Percentual Outliers (%)": perc_outliers,
        "Média com outliers": media_com,
        "Média sem outliers": media_sem,
        "Decisão": decisao
    })

# resultado final em tabela
df_resultado = pd.DataFrame(resultados)

print(df_resultado)

                         Coluna     Q1     Q3   IQR  Limite Inferior  \
0  avg_listening_hours_per_week   7.28  12.68   5.4            -0.82   
1             avg_skips_per_day   8.00  12.00   4.0             2.00   
2             playlists_created   6.00  10.00   4.0             0.00   
3               months_inactive   0.00   2.00   2.0            -3.00   
4                           age  27.00  49.00  22.0            -6.00   

   Limite Superior  Qtd Outliers  Percentual Outliers (%)  Média com outliers  \
0            20.78           166                    0.332            9.988986   
1            18.00           413                    0.826           10.025920   
2            16.00           185                    0.370            8.002680   
3             5.00          2417                    4.834            1.533020   
4            82.00             0                    0.000           38.010280   

   Média sem outliers                Decisão  
0            9.948760  Manter (da

In [43]:
categorical_cols = df.select_dtypes(include=["object"]).columns

def check_real_inconsistencies(df, col):

    original = df[col].astype(str)

    # normalização leve (apenas para comparação interna)
    normalized = original.str.strip().str.lower()

    # categorias reais normalizadas
    unique_map = df.groupby(normalized)[col].unique()

    # inconsistência REAL = mais de 1 escrita para a mesma categoria
    real_inconsistencies = unique_map[unique_map.apply(len) > 1]

    print("\n" + "="*70)
    print(f"Coluna: {col}")

    print(f"Categorias únicas (originais): {original.nunique()}")
    print(f"Categorias únicas (normalizadas): {normalized.nunique()}")

    if len(real_inconsistencies) == 0:
        print("✔ Nenhuma inconsistência REAL detectada")
    else:
        print("⚠ Inconsistências reais encontradas:")
        print(real_inconsistencies)


for col in categorical_cols:
    check_real_inconsistencies(df, col)

C:\Users\mathe\AppData\Local\Temp\ipykernel_3712\2990055707.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns



Coluna: country
Categorias únicas (originais): 12
Categorias únicas (normalizadas): 12
✔ Nenhuma inconsistência REAL detectada

Coluna: signup_date
Categorias únicas (originais): 2982
Categorias únicas (normalizadas): 2982
✔ Nenhuma inconsistência REAL detectada

Coluna: subscription_type
Categorias únicas (originais): 5
Categorias únicas (normalizadas): 5
✔ Nenhuma inconsistência REAL detectada

Coluna: subscription_status
Categorias únicas (originais): 2
Categorias únicas (normalizadas): 2
✔ Nenhuma inconsistência REAL detectada

Coluna: ad_interaction
Categorias únicas (originais): 2
Categorias únicas (normalizadas): 2
✔ Nenhuma inconsistência REAL detectada

Coluna: ad_conversion_to_subscription
Categorias únicas (originais): 2
Categorias únicas (normalizadas): 2
✔ Nenhuma inconsistência REAL detectada

Coluna: favorite_genre
Categorias únicas (originais): 12
Categorias únicas (normalizadas): 12
✔ Nenhuma inconsistência REAL detectada

Coluna: most_liked_feature
Categorias únicas 

<StringArray>
[   'France', 'Indonesia',     'Italy', 'Australia',     'India',   'Germany',
        'UK',    'Brazil',    'Canada',    'Mexico',       'USA',     'Spain']
Length: 12, dtype: str
